# PTCG Replay Triage Starter

Live LB movement is noisy, so I usually look at public replays as debugging data.

This notebook turns public episode JSON files into small review tables:
- broad visible archetype
- visible attacks
- final board summary
- manual loss-label columns

It is meant as a starter table for reviewing repeated loss patterns, not as an automatic deck evaluator.

All features here come from visible public replay JSON only. Hidden cards, private deck notes, and exact rule weights stay out of this notebook.

## Load public episodes

Attach a public daily episode dataset, or upload replay JSON files as an input dataset.

The small `episodes-index` dataset only contains `manifest.csv`; it is useful for finding daily dataset slugs, but it does not contain replay JSON files by itself.

This notebook reads a capped sample of replay JSON files so it can run quickly in a public Kaggle Notebook.

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_JSON_FILES = 80

json_files_all = sorted(INPUT_ROOT.rglob('*.json')) if INPUT_ROOT.exists() else []
json_files = json_files_all[:MAX_JSON_FILES]
manifest_files = sorted(INPUT_ROOT.rglob('manifest.csv')) if INPUT_ROOT.exists() else []

print(f'Found {len(json_files_all)} JSON files under {INPUT_ROOT}')
print(f'Using {len(json_files)} JSON files (MAX_JSON_FILES={MAX_JSON_FILES})')

if manifest_files:
    manifest = pd.read_csv(manifest_files[0])
    print(f'Found manifest: {manifest_files[0]}')
    display(manifest.tail(5))

if not json_files:
    display(pd.DataFrame([
        {
            'status': 'No replay JSON files found',
            'what_to_add': 'Attach a daily episodes dataset such as pokemon-tcg-ai-battle-episodes-2026-07-03, or upload replay JSON files.',
            'output': 'The notebook will still write empty CSV templates to /kaggle/working.',
        }
    ]))
json_files[:5]


Found 5133 JSON files under /kaggle/input
Using 80 JSON files (MAX_JSON_FILES=80)
Found manifest: /kaggle/input/datasets/organizations/kaggle/pokemon-tcg-ai-battle-episodes-index/manifest.csv


,date,daily_dataset_slug,daily_dataset_url,episode_count,total_bytes,top_avg_score,median_avg_score
13,2026-06-29,pokemon-tcg-ai-battle-episodes-2026-06-29,https://www.kaggle.com/datasets/kaggle/pokemon...,5992,21473872622,1411.126738,1031.893209
14,2026-06-30,pokemon-tcg-ai-battle-episodes-2026-06-30,https://www.kaggle.com/datasets/kaggle/pokemon...,5734,21469488677,1386.942931,1112.832961
15,2026-07-01,pokemon-tcg-ai-battle-episodes-2026-07-01,https://www.kaggle.com/datasets/kaggle/pokemon...,5266,21472709170,1344.584468,1180.260904
16,2026-07-02,pokemon-tcg-ai-battle-episodes-2026-07-02,https://www.kaggle.com/datasets/kaggle/pokemon...,5153,21473785185,1274.779486,1135.927808
17,2026-07-03,pokemon-tcg-ai-battle-episodes-2026-07-03,https://www.kaggle.com/datasets/kaggle/pokemon...,5133,21473781654,1250.921561,1104.464801


[PosixPath('/kaggle/input/pokemon-tcg-ai-battle-episodes-2026-07-03/83448419.json'),
 PosixPath('/kaggle/input/pokemon-tcg-ai-battle-episodes-2026-07-03/83448523.json'),
 PosixPath('/kaggle/input/pokemon-tcg-ai-battle-episodes-2026-07-03/83448525.json'),
 PosixPath('/kaggle/input/pokemon-tcg-ai-battle-episodes-2026-07-03/83448526.json'),
 PosixPath('/kaggle/input/pokemon-tcg-ai-battle-episodes-2026-07-03/83448528.json')]

## Extract visible replay features

The parser keeps player 0 and player 1 separate. This matters because both players can expose cards from different archetypes in the same replay.

In [2]:
CARD_HINT_KEYS = {
    'hp', 'cardType', 'energyCards', 'tools', 'attacks',
    'damage', 'preEvolution', 'specialConditions'
}


def walk(value: Any):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from walk(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk(child)


def visible_card_ids(doc: Any, player_index=None) -> Counter[int]:
    counts: Counter[int] = Counter()
    for item in walk(doc):
        if not isinstance(item, dict):
            continue
        if player_index is not None and item.get('playerIndex') != player_index:
            continue
        card_id = item.get('cardId')
        if isinstance(card_id, int):
            counts[card_id] += 1
        generic_id = item.get('id')
        if isinstance(generic_id, int) and CARD_HINT_KEYS.intersection(item.keys()):
            counts[generic_id] += 1
    return counts


def visible_attack_ids(doc: Any, player_index=None) -> Counter[int]:
    counts: Counter[int] = Counter()
    for item in walk(doc):
        if not isinstance(item, dict):
            continue
        if player_index is not None and item.get('playerIndex') != player_index:
            continue
        attack_id = item.get('attackId')
        if isinstance(attack_id, int):
            counts[attack_id] += 1
    return counts


def card_summary(cards) -> str:
    if not cards:
        return ''
    ids = [str(c.get('id', '')) for c in cards if isinstance(c, dict)]
    return ' '.join(ids[:8])


def final_player_state(doc: dict[str, Any], player_index: int) -> dict[str, Any]:
    steps = doc.get('steps') or []
    for pair in reversed(steps):
        for entry in pair or []:
            current = ((entry or {}).get('observation') or {}).get('current') or {}
            players = current.get('players') or []
            if len(players) <= player_index:
                continue
            ps = players[player_index] or {}
            active = ps.get('active') or []
            bench = ps.get('bench') or []
            return {
                'active_ids': card_summary(active),
                'bench_ids': card_summary(bench),
                'bench_count': len([c for c in bench if c]),
                'deck_count': ps.get('deckCount'),
                'hand_count': ps.get('handCount'),
                'discard_count': len(ps.get('discard') or []),
            }
    return {
        'active_ids': '',
        'bench_ids': '',
        'bench_count': 0,
        'deck_count': None,
        'hand_count': None,
        'discard_count': 0,
    }


## Broad archetype buckets

These are broad visible-card hints. Edit this map for your own review.

In [3]:
ARCHETYPE_HINTS = {
    'archaludon_metal': {169, 190, 840, 666},
    'marnie_grimmsnarl': {646, 647, 648, 860},
    'alakazam_psychic': {741, 742, 743, 66, 305},
    'mega_lucario': {677, 678},
    'ogerpon_toolbox': {116, 117, 134, 712, 713, 748, 1051, 1052, 1256},
    'starmie_froslass': {1030, 1031},
    'great_tusk_crustle': {344, 345, 532},
    'hop_trevenant': {288, 289, 299, 304, 307, 308, 309, 310, 878, 879},
    'chandelure_control': {97, 98, 494},
}


def classify_archetype(card_counts: Counter[int]) -> str:
    ids = set(card_counts)
    scored = []
    for name, hints in ARCHETYPE_HINTS.items():
        overlap = len(ids & hints)
        if overlap:
            scored.append((overlap, name))
    return sorted(scored, reverse=True)[0][1] if scored else 'unknown'


## Build review tables

The main table is one row per episode. The archetype and attack tables are quick summaries for scanning.

In [4]:
rows = []
attack_rows = []

for path in json_files:
    try:
        doc = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    if not isinstance(doc, dict) or 'steps' not in doc:
        continue

    p0_card_counts = visible_card_ids(doc, 0)
    p1_card_counts = visible_card_ids(doc, 1)
    p0_attack_counts = visible_attack_ids(doc, 0)
    p1_attack_counts = visible_attack_ids(doc, 1)
    info = doc.get('info') or {}
    rewards = doc.get('rewards') or []
    teams = info.get('TeamNames') or []

    row = {
        'file': str(path),
        'episode_id': info.get('EpisodeId') or path.stem,
        'team_0': teams[0] if len(teams) > 0 else '',
        'team_1': teams[1] if len(teams) > 1 else '',
        'reward_0': rewards[0] if len(rewards) > 0 else None,
        'reward_1': rewards[1] if len(rewards) > 1 else None,
        'p0_bucket': classify_archetype(p0_card_counts),
        'p1_bucket': classify_archetype(p1_card_counts),
        'p0_unique_visible_cards': len(p0_card_counts),
        'p1_unique_visible_cards': len(p1_card_counts),
        'p0_top_attack_ids': ' '.join(str(k) for k, _ in p0_attack_counts.most_common(8)),
        'p1_top_attack_ids': ' '.join(str(k) for k, _ in p1_attack_counts.most_common(8)),
    }
    row.update({f'p0_{k}': v for k, v in final_player_state(doc, 0).items()})
    row.update({f'p1_{k}': v for k, v in final_player_state(doc, 1).items()})
    rows.append(row)

    for player, attack_counts, bucket in [
        (0, p0_attack_counts, row['p0_bucket']),
        (1, p1_attack_counts, row['p1_bucket']),
    ]:
        for attack_id, count in attack_counts.items():
            attack_rows.append({
                'episode_id': row['episode_id'],
                'player': player,
                'team': row[f'team_{player}'],
                'visible_archetype': bucket,
                'attack_id': attack_id,
                'count': count,
            })

episodes = pd.DataFrame(rows)
attack_usage = pd.DataFrame(attack_rows)

episode_cols = [
    'file', 'episode_id', 'team_0', 'team_1',
    'reward_0', 'reward_1',
    'p0_bucket', 'p1_bucket',
    'p0_unique_visible_cards', 'p1_unique_visible_cards',
    'p0_top_attack_ids', 'p1_top_attack_ids',
    'p0_active_ids', 'p0_bench_ids', 'p0_bench_count',
    'p0_deck_count', 'p0_hand_count', 'p0_discard_count',
    'p1_active_ids', 'p1_bench_ids', 'p1_bench_count',
    'p1_deck_count', 'p1_hand_count', 'p1_discard_count',
]
episode_summary = episodes[[c for c in episode_cols if c in episodes.columns]].copy() if not episodes.empty else pd.DataFrame(columns=episode_cols)

if not episodes.empty:
    p0_archetype_count = episodes['p0_bucket'].value_counts().rename_axis('visible_archetype').reset_index(name='player0_count')
    p1_archetype_count = episodes['p1_bucket'].value_counts().rename_axis('visible_archetype').reset_index(name='player1_count')
    archetype_summary = pd.merge(p0_archetype_count, p1_archetype_count, on='visible_archetype', how='outer').fillna(0)
    archetype_summary['total_count'] = archetype_summary['player0_count'] + archetype_summary['player1_count']
    archetype_summary = archetype_summary.sort_values('total_count', ascending=False)
else:
    p0_archetype_count = pd.DataFrame(columns=['visible_archetype', 'player0_count'])
    p1_archetype_count = pd.DataFrame(columns=['visible_archetype', 'player1_count'])
    archetype_summary = pd.DataFrame(columns=['visible_archetype', 'player0_count', 'player1_count', 'total_count'])

if not attack_usage.empty:
    attack_summary = (
        attack_usage
        .groupby(['visible_archetype', 'attack_id'], as_index=False)['count']
        .sum()
        .sort_values('count', ascending=False)
    )
else:
    attack_summary = pd.DataFrame(columns=['visible_archetype', 'attack_id', 'count'])

manual_cols = [
    'episode_id', 'review_player', 'team_0', 'team_1',
    'reward_0', 'reward_1', 'p0_bucket', 'p1_bucket',
    'seat', 'first_real_attacker_turn', 'loss_shape',
    'likely_actionable', 'notes',
]
manual_template = episodes[[c for c in manual_cols if c in episodes.columns]].copy() if not episodes.empty else pd.DataFrame(columns=manual_cols)
for col in manual_cols:
    if col not in manual_template.columns:
        manual_template[col] = ''
manual_template = manual_template[manual_cols]


In [5]:
if episode_summary.empty:
    display(pd.DataFrame([
        {
            'status': 'No episode rows built',
            'reason': 'No replay JSON files were found in /kaggle/input.',
            'next_step': 'Attach one daily episodes dataset or upload replay JSON files, then rerun.',
        }
    ]))
else:
    display(episode_summary.head(20))
    display(p0_archetype_count.head(20))
    display(p1_archetype_count.head(20))
    display(attack_summary.head(30))
    display(manual_template.head(20))


,file,episode_id,team_0,team_1,reward_0,reward_1,p0_bucket,p1_bucket,p0_unique_visible_cards,p1_unique_visible_cards,...,p0_bench_count,p0_deck_count,p0_hand_count,p0_discard_count,p1_active_ids,p1_bench_ids,p1_bench_count,p1_deck_count,p1_hand_count,p1_discard_count
0,/kaggle/input/pokemon-tcg-ai-battle-episodes-2...,83448419,みずあめ,ShumpeiNomura,-1,1,alakazam_psychic,archaludon_metal,19,15,...,4,0,39,8,414,169,1,18,17,14
1,/kaggle/input/pokemon-tcg-ai-battle-episodes-2...,83448523,Yushin Ito,The Debauchery Tea Party,-1,1,starmie_froslass,marnie_grimmsnarl,13,18,...,0,31,4,20,648,646 112,2,29,10,10
2,/kaggle/input/pokemon-tcg-ai-battle-episodes-2...,83448525,tonakaiiii,kazuki0123,1,-1,marnie_grimmsnarl,marnie_grimmsnarl,18,17,...,5,21,6,14,,648 305 112 112,4,25,2,20
3,/kaggle/input/pokemon-tcg-ai-battle-episodes-2...,83448526,tonakaiiii,The Debauchery Tea Party,1,-1,marnie_grimmsnarl,marnie_grimmsnarl,17,18,...,3,24,5,17,,112 112 140 305,4,29,4,19
4,/kaggle/input/pokemon-tcg-ai-battle-episodes-2...,83448528,XP3RiX,btk15049,1,-1,alakazam_psychic,ogerpon_toolbox,20,22,...,3,20,11,18,1052,675 676 116 116 116,5,0,11,33
5,/kaggle/input/pokemon-tcg-ai-battle-episodes-2...,83448530,kazuki0123,The Debauchery Tea Party,1,-1,marnie_grimmsnarl,marnie_grimmsnarl,19,20,...,5,12,6,24,,140 112 112,3,4,7,40
6,/kaggle/input/pokemon-tcg-ai-battle-episodes-2...,83448536,btk15049,kazuki0123,-1,1,ogerpon_toolbox,marnie_grimmsnarl,22,19,...,4,6,6,32,648,112 112 648 112 646,5,8,12,21
7,/kaggle/input/pokemon-tcg-ai-battle-episodes-2...,83448539,Jack,Akira-Ninth,-1,1,archaludon_metal,mega_lucario,14,17,...,1,20,4,31,678,675 676 678 675 676,5,13,7,22
8,/kaggle/input/pokemon-tcg-ai-battle-episodes-2...,83448542,kashiwashira,【ＡＩと共に、ＡＩと戦う】tubotu,1,-1,unknown,alakazam_psychic,19,24,...,5,13,14,21,,858 66 741 66,4,0,31,20
9,/kaggle/input/pokemon-tcg-ai-battle-episodes-2...,83448544,tonakaiiii,atom1231,1,-1,marnie_grimmsnarl,great_tusk_crustle,18,19,...,5,7,5,28,58,,0,6,3,43


,visible_archetype,player0_count
0,alakazam_psychic,21
1,marnie_grimmsnarl,12
2,ogerpon_toolbox,11
3,starmie_froslass,9
4,archaludon_metal,7
5,great_tusk_crustle,6
6,mega_lucario,5
7,unknown,4
8,chandelure_control,4
9,hop_trevenant,1


,visible_archetype,player1_count
0,alakazam_psychic,30
1,marnie_grimmsnarl,18
2,unknown,8
3,ogerpon_toolbox,7
4,mega_lucario,6
5,archaludon_metal,4
6,great_tusk_crustle,3
7,chandelure_control,3
8,starmie_froslass,1


,visible_archetype,attack_id,count
3,alakazam_psychic,1072,2301
14,marnie_grimmsnarl,937,1221
6,archaludon_metal,253,412
33,unknown,716,365
0,alakazam_psychic,716,362
24,ogerpon_toolbox,716,332
1,alakazam_psychic,1070,286
19,mega_lucario,983,227
28,starmie_froslass,1487,179
23,ogerpon_toolbox,371,176


,episode_id,review_player,team_0,team_1,reward_0,reward_1,p0_bucket,p1_bucket,seat,first_real_attacker_turn,loss_shape,likely_actionable,notes
0,83448419,,みずあめ,ShumpeiNomura,-1,1,alakazam_psychic,archaludon_metal,,,,,
1,83448523,,Yushin Ito,The Debauchery Tea Party,-1,1,starmie_froslass,marnie_grimmsnarl,,,,,
2,83448525,,tonakaiiii,kazuki0123,1,-1,marnie_grimmsnarl,marnie_grimmsnarl,,,,,
3,83448526,,tonakaiiii,The Debauchery Tea Party,1,-1,marnie_grimmsnarl,marnie_grimmsnarl,,,,,
4,83448528,,XP3RiX,btk15049,1,-1,alakazam_psychic,ogerpon_toolbox,,,,,
5,83448530,,kazuki0123,The Debauchery Tea Party,1,-1,marnie_grimmsnarl,marnie_grimmsnarl,,,,,
6,83448536,,btk15049,kazuki0123,-1,1,ogerpon_toolbox,marnie_grimmsnarl,,,,,
7,83448539,,Jack,Akira-Ninth,-1,1,archaludon_metal,mega_lucario,,,,,
8,83448542,,kashiwashira,【ＡＩと共に、ＡＩと戦う】tubotu,1,-1,unknown,alakazam_psychic,,,,,
9,83448544,,tonakaiiii,atom1231,1,-1,marnie_grimmsnarl,great_tusk_crustle,,,,,


## Manual loss labels

The exported template leaves the final loss label manual. These labels are usually enough for quick review:

- setup miss
- tempo loss
- no backup attacker
- bench-snipe collapse
- wrong target priority
- resource exhaustion
- deck-out / low-deck risk
- probably variance

After labeling a few losses, I test one narrow local change at a time instead of reacting to a single replay.

This notebook only uses public replay information. Exact deck choices, rule weights, and matchup notes can stay in a separate private review sheet.

## Export CSVs

In [6]:
episode_summary.to_csv(OUTPUT_DIR / 'episode_summary.csv', index=False)
archetype_summary.to_csv(OUTPUT_DIR / 'archetype_summary.csv', index=False)
attack_summary.to_csv(OUTPUT_DIR / 'attack_usage_summary.csv', index=False)
manual_template.to_csv(OUTPUT_DIR / 'manual_loss_label_template.csv', index=False)

for name in [
    'episode_summary.csv',
    'archetype_summary.csv',
    'attack_usage_summary.csv',
    'manual_loss_label_template.csv',
]:
    print(OUTPUT_DIR / name)


/kaggle/working/episode_summary.csv
/kaggle/working/archetype_summary.csv
/kaggle/working/attack_usage_summary.csv
/kaggle/working/manual_loss_label_template.csv
